<a href="https://colab.research.google.com/github/imchrisrueda/extML/blob/main/k_brazos/banditUCB.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Bandido de k-brazos — UCB (UCB1 y UCB2)

**Descripción:** En este notebook se realiza un estudio comparativo de algoritmos de la familia **UCB (Upper Confidence Bound)** para el problema del **bandido de k-brazos**.

## Algoritmos evaluados e hiperparámetros

### UCB1 (Auer et al., 2002)

Se evalúa **UCB1** variando la constante de exploración **c**:

$\displaystyle \mathrm{UCB}(a)=Q(a)+c\sqrt{\frac{\ln t}{N(a)}}$

Valores probados:
- $c \in \{1,\sqrt{2},2\}$

### UCB2 (Auer et al., 2002)

Se evalúa **UCB2** variando el parámetro **$\alpha$**, que controla el crecimiento de las **épocas**.

Índice de selección:
$\displaystyle \mathrm{UCB2}(a)=Q(a)+a_{n,r_a}$

Término de confianza:
$\displaystyle a_{n,r}=\sqrt{\frac{(1+\alpha)\,\ln\!\left(\frac{e\,n}{\tau(r)}\right)}{2\,\tau(r)}}$

Tamaño de épocas:
$\displaystyle \tau(r)=\left\lceil(1+\alpha)^r\right\rceil$

Valores probados:
- $\alpha \in \{0.1, 0.5, 0.9\}$, con la restricción $0<\alpha<1$.

## Preparación del entorno

In [2]:
# Copiar el repositorio.

# En Google Colab (ejemplo):
# !git clone https://github.com/imchrisrueda/extML.git
# %cd /content/extML/k_brazos


In [3]:
# Importamos todas las clases y funciones

import sys
from pathlib import Path
from typing import List

import numpy as np

# Añadir directorios al path de Python 
sys.path.append('/content/eml_k_bandit')
sys.path.append('/content/eml_k_bandit/k_brazos')

cwd = Path.cwd()
for candidate in (cwd, cwd / 'k_brazos'):
    candidate_str = str(candidate)
    if candidate_str not in sys.path and candidate.exists():
        sys.path.append(candidate_str)

# Clases y funciones para UCB1 y 2 y Bandits
from src.algorithms import Algorithm, UCB1, UCB2
from src.arms import ArmNormal, ArmBinomial, ArmBernoulli, Bandit
from src.plotting import plot_average_rewards, plot_optimal_selections, plot_arm_statistics, plot_regret


## Experimento

La función `run_experiment(...)` implementa el bucle principal de simulación para una instancia de bandido y un conjunto de algoritmos. En cada ejecución (`run`) se reinicia el estado de los algoritmos, se simulan `steps` interacciones y se acumulan las métricas relevantes.

Para asegurar **reproducibilidad**, se fija una **semilla** (por ejemplo con `np.random.seed(...)`) antes de ejecutar los experimentos.

En este notebook, `run_experiment(...)` se utiliza para comparar configuraciones de la familia **UCB** (UCB1 con distintos valores de **c** y UCB2 con distintos valores de **α**) sobre tres distribuciones de recompensa: **Normal**, **Bernoulli** y **Binomial**.

Las salidas del experimento son:

- `rewards`: recompensa promedio por paso y algoritmo.
- `optimal_selections`: proporción de veces que se selecciona el brazo óptimo en cada paso.
- `regrets`: regret acumulado empírico por algoritmo.
- `counts_accum` y `values_accum`: estadísticas agregadas por brazo para análisis posteriores.

In [4]:
def run_experiment(bandit: Bandit, algorithms: List[Algorithm], steps: int, runs: int):

    optimal_arm = bandit.optimal_arm  # Necesario para calcular el porcentaje de selecciones óptimas.

    rewards = np.zeros((len(algorithms), steps)) # Matriz para almacenar las recompensas promedio.

    optimal_selections = np.zeros((len(algorithms), steps))  # Matriz para almacenar el porcentaje de selecciones óptimas.

    np.random.seed(seed)  # Fijamos semilla para que las ejecuciones sean reproducibles

    #para el regret
    regrets = np.zeros((len(algorithms), steps)) # Matriz para almacenar el arrepentimiento
    q = bandit.get_expected_value(optimal_arm) # Recompensa del brazo óptimo
    # Lo voy acumulando para mostrarlo en el gráfico
    counts_accum = np.zeros((len(algorithms), bandit.k))
    values_accum = np.zeros((len(algorithms), bandit.k))


    for run in range(runs):
        current_bandit = Bandit(arms=bandit.arms)

        for algo in algorithms:
            algo.reset() # En cada corrida reiniciamos los algoritmos para no mezclar experiencia entre runs.

        total_regret = np.zeros(len(algorithms)) # Acumulador de arrepentimiento por algoritmo

        for step in range(steps):
            for idx, algo in enumerate(algorithms):
                chosen_arm = algo.select_arm() # Seleccionar un brazo según la política del algoritmo.
                reward = current_bandit.pull_arm(chosen_arm) # Obtener la recompensa del brazo seleccionado.
                algo.update(chosen_arm, reward) # Actualizar el valor estimado del brazo seleccionado.

                rewards[idx, step] += reward # Acumular la recompensa obtenida en la matriz rewards para el algoritmo idx en el paso step.
                
                #modificar optimal_selections cuando el brazo elegido se corresponda con el brazo óptimo optimal_arm
                if chosen_arm == optimal_arm: # Si se ha elegido el brazo óptimo...
                    optimal_selections[idx, step] += 1 # ... añadir uno a la acumulación de selecciones de brazos óptimos


                total_regret[idx] += (q - reward) # El arrepentimiento es la q - la recompensa  -> lo que hubiéramos obtenido eligiendo el mejor brazo
                regrets[idx, step] += total_regret[idx] # Actualizar el regret acumulado


        for idx, algo in enumerate(algorithms):
            counts_accum[idx] += algo.counts
            values_accum[idx] += algo.values
            
    #calcular el porcentaje de selecciones óptimas y almacenar en optimal_selections
    rewards /= runs
    optimal_selections /= runs
    regrets /= runs

    counts_accum /= runs
    values_accum /= runs


    return rewards, optimal_selections, regrets, counts_accum, values_accum


## Ejecución del experimento

Se ejecuta el estudio con los parámetros `k=10`, `steps=1000` y `runs=500`, usando una semilla fija (`seed=42`) para asegurar reproducibilidad. La comparación se realiza sobre tres tipos de bandido:

- **Normal** ($N(\mu, \sigma^2)$ con $\sigma=1$ en la implementación actual).
- **Bernoulli** (recompensas binarias `0/1`).
- **Binomial** (`n=10`).

En cada distribución se comparan configuraciones de la familia UCB:

### UCB1
Se evalúa **UCB1** variando la constante de exploración $c$:
- $c=1$
- $c=\sqrt{2}$ (caso clásico)
- $c=2$

### UCB2
Se evalúa **UCB2** variando el parámetro $\alpha$ (con restricción $0<\alpha<1$):
- $\alpha=0.1$
- $\alpha=0.5$
- $\alpha=0.9$

Se analiza cómo los hiperparámetros $c$ y $\alpha$ afectan al equilibrio exploración–explotación y cómo cambia el desempeño de UCB1 y UCB2 según la naturaleza de la distribución de recompensas.

In [8]:
# Parámetros del experimento y construcción de algoritmos UCB1 y UCB2

seed = 42
np.random.seed(seed)

k = 10
steps = 1000
runs = 500

def build_ucb_algorithms(k: int):
    return [
        UCB1(k=k, c=1.0),
        UCB1(k=k, c=np.sqrt(2.0)),
        UCB1(k=k, c=2.0),
        UCB2(k=k, alpha=0.1),
        UCB2(k=k, alpha=0.5),
        UCB2(k=k, alpha=0.9)
    ]


In [9]:
# Crear bandidos (Normal, Bernoulli, Binomial) y ejecutar experimentos

bandit_normal = Bandit(arms=ArmNormal.generate_arms(k))
bandit_bernoulli = Bandit(arms=ArmBernoulli.generate_arms(k))
bandit_binomial = Bandit(arms=ArmBinomial.generate_arms(k, n=10))

algorithms_normal = build_ucb_algorithms(k)
algorithms_bernoulli = build_ucb_algorithms(k)
algorithms_binomial = build_ucb_algorithms(k)

rewards_normal, optimal_normal, regrets_normal, counts_normal, values_normal = run_experiment(bandit_normal, algorithms_normal, steps, runs)
rewards_bernoulli, optimal_bernoulli, regrets_bernoulli, counts_bernoulli, values_bernoulli = run_experiment(bandit_bernoulli, algorithms_bernoulli, steps, runs)
rewards_binomial, optimal_binomial, regrets_binomial, counts_binomial, values_binomial = run_experiment(bandit_binomial, algorithms_binomial, steps, runs)


g:\Mi unidad\02 - Máster en IA Murcia\Segundo cuatrimestre\Extensiones de ML\Tarea 1\extML-master\k_brazos\src\algorithms\ucb.py:112: RuntimeWarning: invalid value encountered in sqrt
  return float(np.sqrt(((1.0 + self.alpha) * np.log((np.e * n) / tau_r)) / (2.0 * tau_r)))


In [10]:
# Verificar shapes para las 3 distribuciones (si ya están calculadas)

def print_shapes(name, rewards, optimal, regrets, counts, values):
    print(f"\n=== {name} ===")
    print("rewards:", rewards.shape)
    print("optimal:", optimal.shape)
    print("regrets:", regrets.shape)
    print("counts:", counts.shape)
    print("values:", values.shape)

print_shapes("Normal", rewards_normal, optimal_normal, regrets_normal, counts_normal, values_normal)
print_shapes("Bernoulli", rewards_bernoulli, optimal_bernoulli, regrets_bernoulli, counts_bernoulli, values_bernoulli)
print_shapes("Binomial", rewards_binomial, optimal_binomial, regrets_binomial, counts_binomial, values_binomial)


=== Normal ===
rewards: (6, 1000)
optimal: (6, 1000)
regrets: (6, 1000)
counts: (6, 10)
values: (6, 10)

=== Bernoulli ===
rewards: (6, 1000)
optimal: (6, 1000)
regrets: (6, 1000)
counts: (6, 10)
values: (6, 10)

=== Binomial ===
rewards: (6, 1000)
optimal: (6, 1000)
regrets: (6, 1000)
counts: (6, 10)
values: (6, 10)


### Brazo óptimo y separación respecto al segundo mejor brazo

In [11]:
def resumen_optimo(bandit, nombre: str):
    esperados = np.array([bandit.get_expected_value(i) for i in range(bandit.k)])
    orden = np.argsort(esperados)[::-1]  # de mayor a menor
    best, second = int(orden[0]), int(orden[1])
    gap = float(esperados[best] - esperados[second])

    print(f"- **{nombre}**: brazo óptimo índice `{best}` (1-based: `{best+1}`), "
          f"valor esperado `*={esperados[best]:.2f}`. "
          f"Segundo mejor: índice `{second}` con valor esperado `{esperados[second]:.2f}` "
          f"(gap ≈ `{gap:.2f}`).")

print("### Brazo óptimo y separación respecto al segundo mejor brazo\n")
resumen_optimo(bandit_normal, "Normal")
resumen_optimo(bandit_bernoulli, "Bernoulli")
resumen_optimo(bandit_binomial, "Binomial (n=10)")

### Brazo óptimo y separación respecto al segundo mejor brazo

- **Normal**: brazo óptimo índice `7` (1-based: `8`), valor esperado `*=9.56`. Segundo mejor: índice `6` con valor esperado `8.80` (gap ≈ `0.76`).
- **Bernoulli**: brazo óptimo índice `0` (1-based: `1`), valor esperado `*=0.96`. Segundo mejor: índice `1` con valor esperado `0.83` (gap ≈ `0.13`).
- **Binomial (n=10)**: brazo óptimo índice `2` (1-based: `3`), valor esperado `*=7.80`. Segundo mejor: índice `8` con valor esperado `6.10` (gap ≈ `1.70`).


A continuación se reporta, para cada distribución, el **brazo óptimo** (mayor valor esperado), el **segundo mejor** y la separación **gap** (diferencia entre ambos valores esperados).  
Estos valores se usarán luego para interpretar los resultados de las gráficas.

- **Normal**: óptimo índice `7` (8.º en notación 1-based), valor esperado `μ*=9.56`. Segundo mejor índice `6`, `μ=8.80` (gap ≈ `0.76`).
- **Bernoulli**: óptimo índice `0` (1.º en notación 1-based), valor esperado `p*=0.96`. Segundo mejor índice `1`, `p=0.83` (gap ≈ `0.13`).
- **Binomial (n=10)**: óptimo índice `2` (3.º en notación 1-based), valor esperado `E[X]=7.80`. Segundo mejor índice `8`, `E[X]=6.10` (gap ≈ `1.70`).

En general, un **gap mayor** implica que debería ser más fácil para los algoritmos concentrarse en el brazo óptimo con menos exploración, mientras que un **gap menor** hace más difícil distinguir el mejor brazo y puede aumentar el tiempo de exploración necesario.

### Función auxiliar `build_arm_stats`

Para generar el gráfico `plot_arm_statistics` se utiliza la función auxiliar `build_arm_stats`, que construye la estructura de datos necesaria a partir de las salidas `counts_accum` y `values_accum` del experimento.

Esta función resume, para cada algoritmo y cada brazo:

- número promedio de selecciones (`selected`),
- recompensa media estimada (`avg_reward`),
- e indicador de si el brazo es óptimo (`is_optimal`).